# Cooper mail playground

Send and read test emails against the local stack (`docker compose up -d`). Standard library only.

- **GreenMail** (SMTP `3025`, IMAP `3143`): the server a Mailbox polls for ticket intake. Auth is disabled; the login is the full address, any password. Sending to an address creates its inbox.
- **Mailpit** (SMTP `1025`, UI <http://localhost:8025>): captures mail Cooper sends out (replies, notifications). It relays every message on to GreenMail.
- **Roundcube** (<http://localhost:8026>): webmail on GreenMail. Sign in as any address to write to a Mailbox and read Cooper's replies.

Ports match `.env.example`.

In [1]:
import email
import imaplib
import json
import os
import smtplib
import urllib.request
from pathlib import Path
from email.message import EmailMessage
from email.utils import make_msgid


def load_env(name=".env"):
    """Load KEY=VALUE lines from the nearest .env upwards; real environment variables win."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        path = directory / name
        if path.is_file():
            for line in path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, value = line.partition("=")
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
            return path
    return None

print("Loaded", load_env() or "no .env found (using defaults)")

HOST = "localhost"
GREENMAIL_SMTP = int(os.getenv("GREENMAIL_SMTP_PORT", 3025))
GREENMAIL_IMAP = int(os.getenv("GREENMAIL_IMAP_PORT", 3143))
MAILPIT_SMTP = int(os.getenv("MAILPIT_SMTP_PORT", 1025))
MAILPIT_UI = int(os.getenv("MAILPIT_UI_PORT", 8025))

# The address of the Mailbox configured in Cooper (Administration -> Mailboxes).
MAILBOX = "support@example.com"

Loaded /home/subanov/work/janea/cooper/.env


## Send an email into a Mailbox (ticket intake)

In [2]:
def send(to=MAILBOX, sender="alice@acmetest.com", subject="Software problem",
         body="Another problem with new software.", port=GREENMAIL_SMTP,
         in_reply_to=None, attachments=()):
    msg = EmailMessage()
    msg["From"], msg["To"], msg["Subject"] = sender, to, subject
    msg["Message-ID"] = make_msgid(domain="example.com")
    if in_reply_to:  # thread onto an existing conversation
        msg["In-Reply-To"] = msg["References"] = in_reply_to
    msg.set_content(body)
    for name, data, mime in attachments:
        main, sub = mime.split("/")
        msg.add_attachment(data, maintype=main, subtype=sub, filename=name)
    with smtplib.SMTP(HOST, port) as smtp:
        smtp.send_message(msg)
    return msg["Message-ID"]

message_id = send()
message_id

'<179508279117.242538.8962077570994712680@example.com>'

To test SDREF threading, send again with the ticket reference in the subject, e.g.
`send(subject="Re: [SDREF-123] Test ticket", in_reply_to=message_id)`. Use the real reference Cooper generated.

## Read a Mailbox over IMAP (GreenMail)

In [ ]:
def read_imap(user=MAILBOX, folder="INBOX", limit=10):
    with imaplib.IMAP4(HOST, GREENMAIL_IMAP) as imap:
        imap.login(user, "any")
        imap.select(folder, readonly=True)
        _, data = imap.search(None, "ALL")
        for num in data[0].split()[-limit:]:
            _, parts = imap.fetch(num, "(RFC822)")
            m = email.message_from_bytes(parts[0][1])
            body = m.get_body(preferencelist=("plain", "html"))
            print(f"#{num.decode()} {m['From']} -> {m['To']} | {m['Subject']}")
            print("   ", (body.get_content().strip() if body else "")[:200].replace("\n", " "))

read_imap()

## Read mail Cooper sent (Mailpit)

In [ ]:
def mailpit(path):
    with urllib.request.urlopen(f"http://{HOST}:{MAILPIT_UI}/api/v1/{path}") as r:
        return json.load(r)

def read_mailpit(limit=10):
    for m in mailpit(f"messages?limit={limit}")["messages"]:
        to = ", ".join(a["Address"] for a in m["To"])
        print(f"{m['Created']} {m['From']['Address']} -> {to} | {m['Subject']}")
        print("   ", m["Snippet"])

read_mailpit()

## Send directly to Mailpit (check how mail renders)

In [ ]:
send(to="analyst@example.com", subject="Mailpit test", port=MAILPIT_SMTP)
read_mailpit(3)